# Modélisation de courbes de demande de détail non linéaires avec PROC GAM

## Résumé

Ce notebook utilise **PROC GAM** pour modéliser les ventes hebdomadaires en unités d'un magasin d'alimentation comme une fonction lisse et non linéaire du prix en rayon, de la température du magasin (un indicateur de saisonnalité) et des dépenses promotionnelles, avec un effet paramétrique pour l'indicateur de promotion. Les modèles additifs généralisés permettent à un category manager de retrouver les véritables formes courbes d'élasticité-prix et de demande saisonnière qu'une régression linéaire ne pourrait pas détecter, ce qui permet des décisions de tarification et de promotion plus fines.

## Sources de données

| Table | Lignes | Granularité | Variables clés | Description |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | semaine | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Série hebdomadaire synthétique de point de vente pour un magasin d'alimentation phare sur 100 semaines consécutives (près de deux cycles saisonniers). Générée en ligne avec `call streaminit(20250531)` et `rand()`. Les ventes hebdomadaires en unités suivent un processus de demande de Poisson dont la moyenne est déterminée par une courbe de réponse-prix exponentielle, un effet quadratique de température/saisonnalité culminant près de 72°F, et une hausse concave (racine carrée) des dépenses promotionnelles, plus un indicateur discret de semaine promotionnelle. |

# Modélisation de courbes de demande de détail non linéaires avec PROC GAM

La demande de détail répond rarement au prix, à la météo ou aux dépenses promotionnelles de façon linéaire. Une petite baisse de prix sur un produit de base peut à peine faire bouger le volume, alors que le franchissement d'un seuil psychologique de prix peut déclencher un bond brutal ; la demande liée à la météo culmine dans une plage intermédiaire confortable et retombe aux deux extrêmes ; et la hausse promotionnelle montre des rendements décroissants à mesure que les dépenses augmentent.

**PROC GAM** (modèles additifs généralisés) ajuste chaque facteur avec son propre terme de spline lisse, de sorte que ce sont les données — et non une hypothèse linéaire rigide — qui déterminent la forme de chaque courbe de demande. Nous modélisons ici les ventes hebdomadaires en unités d'un magasin d'alimentation phare sur 100 semaines consécutives, en combinant un indicateur de promotion paramétrique avec des splines de lissage sur le prix, la température et les dépenses promotionnelles, sous une réponse de Poisson.

## Étape 1 — Générer une série de ventes hebdomadaires synthétique

Nous simulons 100 semaines consécutives (près de deux cycles saisonniers) de données de point de vente pour un magasin phare. Le processus générateur de données est délibérément non linéaire afin de vérifier que GAM retrouve des formes réalistes :

- **Price** entraîne le volume via une courbe de réponse exponentielle (`exp(-1.1 * Price)`), de sorte que la demande augmente fortement lorsque le prix baisse.
- **Temperature** sert d'indicateur de saisonnalité avec un pic quadratique proche de 72°F, modélisant une fréquentation liée au confort.
- **PromoSpend** apporte une hausse concave en racine carrée (rendements décroissants).
- Un indicateur discret **Promotion** ajoute un effet en palier paramétrique lors des semaines promues.

Les `Units` hebdomadaires sont tirées d'une distribution de Poisson, conformément à la nature de comptage des ventes en unités.

In [1]:
DONNÉES store_sales;
   APPELER streaminit(20250531);
   LONGUEUR Promotion $3;
   FAIRE Week = 1 JUSQU_À 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      SI rand("uniform") < 0.28 ALORS FAIRE;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      FIN;
      SINON FAIRE;
         Promotion  = "No";
         PromoSpend = 0;
      FIN;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      SORTIE;
   FIN;
EXÉCUTER;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Étape 2 — Profiler les données simulées

Un rapide `PROC MEANS` confirme que les variables couvrent des plages plausibles pour le commerce de détail avant la modélisation : les comptages d'unités sont des entiers non négatifs, le prix se regroupe autour de quelques dollars, la température balaie un cycle saisonnier complet, et les dépenses promotionnelles sont nulles les semaines sans promotion.

In [2]:
PROCÉDURE MOYENNES DONNÉES=store_sales n mean MIN MAX maxdec=2;
   VAR Units Price Temperature PromoSpend;
   ÉTIQUETTE Units="Ventes (unités)" Price="Prix ($)" Temperature="Température (°F)"
         PromoSpend="Dépense promo. ($)";
EXÉCUTER;

                                                  The MEANS Procedure

 Variable     Label                       N           Mean     Minimum     Maximum
 ---------------------------------------------------------------------------------
 Units        Ventes (unités)           100           7.03        0.00      103.00
 Price        Prix ($)                  100           3.15        2.81        3.48
 Temperature  Température (°F)          100          55.50       22.72       83.49
 PromoSpend   Dépense promo. ($)        100         128.76        0.00      779.00
 ---------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Étape 3 — Ajuster le modèle additif complet de la demande

Le modèle central combine :

- `param(Promotion)` — un effet paramétrique (linéaire) pour l'indicateur de semaine promotionnelle, déclaré dans l'instruction `CLASS`.
- `spline(Price, df=4)` — une spline de lissage cubique capturant la réponse-prix courbe.
- `spline(Temperature, df=4)` — une courbe de saisonnalité lisse.
- `spline(PromoSpend, df=3)` — la hausse promotionnelle à rendements décroissants.

Comme les ventes en unités sont des comptages, nous spécifions `dist=poisson` (lien log). `method=gcv` permet à la validation croisée généralisée de guider la souplesse de chaque composante sans imposer explicitement de degrés de liberté. L'instruction `OUTPUT` écrit les prédictions et résidus par observation dans `gam_fit`.

La procédure rapporte une **déviance de 43.59** contre une **déviance nulle de 2041.12** — les quatre facteurs lisses et paramétriques expliquent presque toute la variation qu'un modèle à constante seule laisserait de côté — et un **AIC de 254.61**. L'estimation paramétrique `PROMOTIONYES` est positive (+0.41 sur l'échelle logarithmique), confirmant la hausse de volume promotionnelle, et la spline de prix porte une tendance linéaire fortement négative (−1.71), la signature d'une demande décroissante avec le prix.

In [3]:
PROCÉDURE gam DONNÉES=store_sales PLOTS=components(CLM commonaxes);
   CLASSE Promotion;
   MODÈLE Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   ÉTIQUETTE Units="Ventes (unités)" Promotion="Promotion" Price="Prix ($)"
         Temperature="Température (°F)" PromoSpend="Dépense promo. ($)";
   SORTIE out=gam_fit predicted residual;
EXÉCUTER;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Ventes (unités)
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Prix ($))                 4.0000 


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Étape 4 — Un modèle centré sur le prix et la saisonnalité

Pour une revue tarifaire plus légère, nous réajustons avec seulement les deux facteurs lisses les plus pertinents pour la décision — le prix et la température — en donnant au prix une flexibilité supplémentaire (`df=5`) pour résoudre tout coude près d'un seuil psychologique de prix. L'indicateur de promotion est conservé comme contrôle paramétrique.

Retirer les dépenses promotionnelles fait passer la **déviance à 62.80** et l'**AIC à 269.82**, toutes deux supérieures aux 43.59 et 254.61 du modèle complet. Le terme paramétrique `PROMOTIONYES` absorbe également ici davantage du signal promotionnel (+1.73 contre +0.41), puisque la spline de dépenses n'est plus présente pour le porter. La spline de prix conserve sa tendance négative (−1.51), donc l'histoire centrale de la demande reste stable d'une spécification à l'autre.

In [4]:
PROCÉDURE gam DONNÉES=store_sales PLOTS=components(CLM);
   CLASSE Promotion;
   MODÈLE Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   ÉTIQUETTE Units="Ventes (unités)" Promotion="Promotion" Price="Prix ($)"
         Temperature="Température (°F)";
EXÉCUTER;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Ventes (unités)
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Prix ($))                 5.0000         5.0000
Spline(Température (°F))         4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interprétation des résultats

La table **Regression Model Analysis** rapporte la tendance linéaire de chaque composante ainsi que l'effet promotionnel paramétrique. Le coefficient positif `PROMOTIONYES` (+0.41 dans le modèle complet, +1.73 dans le modèle allégé) confirme la hausse de volume promotionnelle attendue, tandis que la tendance linéaire négative sur la spline de prix (−1.71 et −1.51) reflète une demande classique décroissante avec le prix. Le petit terme linéaire positif de la spline de température (+0.03) est attendu : sa véritable histoire est la courbure autour du pic de confort à 72°F, qu'un unique coefficient linéaire ne peut résumer.

La table **Smoothing Model Analysis** rapporte les degrés de liberté consommés par chaque spline. Le prix et la température utilisent chacun 4 DF effectifs (5 pour le prix dans le modèle allégé) et les dépenses promotionnelles 3 — chacun bien au-dessus du DF unique qu'utiliserait une droite, ce qui explique précisément pourquoi une régression linéaire manquerait ces relations courbes.

La table **Fit Statistics** permet de comparer directement les deux spécifications. Le modèle complet à quatre facteurs affiche une déviance de 43.59 et un AIC de 254.61 contre 62.80 et 269.82 pour le modèle allégé ; les deux critères favorisent le modèle complet, montrant que les dépenses promotionnelles et l'indicateur de promotion ajoutent un pouvoir explicatif au-delà du prix et de la température seuls. Par rapport à la déviance nulle de 2041.12, les deux modèles capturent l'écrasante majorité de la variation de la demande.

Pris ensemble, ces tables donnent à un category manager une histoire de demande quantifiée et fondée sur les données : une réponse-prix marquée qui éclaire la profondeur des démarques, une fenêtre saisonnière liée à la température, et un effet de dépenses promotionnelles à rendements décroissants — un guide bien plus précis qu'une simple estimation d'élasticité linéaire. (`PROC GAM` accepte aussi `plots=components` pour tracer les courbes de prédiction partielle de chaque terme lisse ; les tables numériques de composantes ci-dessus sont la source à partir de laquelle ces courbes sont tracées.)